In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re


In [ ]:
url = 'https://raw.githubusercontent.com/spamella04/Restaurant_and_Market_Health_Inspections/refs/heads/main/Restaurant_and_Market_Health_Inspections%20(1).csv'
data = pd.read_csv(url)

In [ ]:
data.shape

In [ ]:
data.head()

In [ ]:
data.info()

In [ ]:
data.duplicated().sum()

In [ ]:
data.isnull().sum()

In [ ]:
min_max_scores = data.groupby('grade')['score'].agg(['min', 'max']).reset_index()

# Mostrar el resultado
print(min_max_scores)

In [ ]:
#Imprimiendo los registros donde el grade es null
print(data[data['grade'].isnull()])

In [ ]:
print(data[data['program_name'].isnull()])

In [ ]:
#Eliminandos los registros donde program_name es null
data = data.dropna(subset=['program_name'])

In [ ]:
#Remplazando con el valor de 'C' los valores nulos de la columna grade
data['grade'].fillna('C', inplace=True)

In [ ]:
#convertir la columna fecha a date
data['activity_date'] = pd.to_datetime(data['activity_date'], errors='coerce').dt.date

In [ ]:
#Creacion de nuevas columnas: year, month y day



In [ ]:
data['activity_date'] = pd.to_datetime(data['activity_date'], errors='coerce')
data['year'] = data['activity_date'].dt.year
data['month'] = data['activity_date'].dt.month
data['day'] = data['activity_date'].dt.day

In [ ]:
#Eliminado columna activity_date
data.drop('activity_date', axis=1, inplace=True)

In [ ]:
#Creacion de nuevas columnas: PE DESCRIPTION, categoriza los establecimientos según su tipo, tamaño y nivel de riesgo. "FOOD MKT RETAIL (1-1,999 SF) LOW RISK".
#category, size, riskLevel


In [ ]:
# Función para separar la información
def categorize_description(desc):
    # Valores predeterminados
    category = "Sin Información"
    size = "Desconocido"
    risk = "Desconocido"

    # Formato estándar
    standard_format = re.match(r'(.+?) \((.*?)\)( SEATS)?\s*(.+)', desc)
    if standard_format:
        category = standard_format.group(1).strip()
        size = standard_format.group(2).strip() + (" SEATS" if standard_format.group(3) else "")
        risk = standard_format.group(4).strip()
    else:
        # Formatos no estándar
        if "CATERER" in desc:
            category = "CATERER"
            size_match = re.search(r'\((.*?)\)', desc)
            if size_match:
                size = size_match.group(1)
        elif "FOOD MARKET" in desc:
            category = "FOOD MARKET"
        elif "FOOD PROCESSING" in desc:
            category = "FOOD PROCESSING"
            size_match = re.search(r'\((.*?)\)', desc)
            if size_match:
                size = size_match.group(1)
        elif "FOOD WAREHOUSE" in desc:
            category = "FOOD WAREHOUSE"
            size_match = re.search(r'\((.*?)\)', desc)
            if size_match:
                size = size_match.group(1)
        elif "SENIOR FEEDING SITE" in desc:
            category = "SENIOR FEEDING SITE"
        elif "PRIVATE SCHOOL" in desc:
            category = "PRIVATE SCHOOL"
        elif "WHOLESALE FOOD COMPLEX" in desc:
            category = "WHOLESALE FOOD COMPLEX"
    
    return pd.Series([category, size, risk])

In [ ]:
# Aplicar la función al DataFrame
data[['category', 'size', 'riskLevel']] = data['pe_description'].apply(categorize_description)

In [ ]:
#Eliminando registros donde riskLevel es Desconocido
data = data[data['riskLevel'] != 'Desconocido']

In [ ]:
data = data[data['riskLevel'] != 'SQ. FT.']

In [ ]:
data.head()

In [ ]:
#Eliminando registros con especificos serial_number
data = data[data['serial_number'] != 'DAVTNZVG3']
data = data[data['serial_number'] != 'DADU3VZ4L']
data = data[data['serial_number'] != 'DAGNSISLK']
data = data[data['serial_number'] != 'DAOJO74T3']


In [ ]:
data.isnull().sum()

In [ ]:
#Eliminando columna pe_description
data.drop('pe_description', axis=1, inplace=True)

In [ ]:
#Deteccion de outliers en la columna score
plt.boxplot(data['score'])
plt.show()

In [ ]:
#Columnas a utilizarenel análisis

In [ ]:
columns = ['serial_number','year','month','day','facility_id','facility_name', 'facility_zip','score','grade','service_description','category','size','riskLevel','owner_id','owner_name',]

In [ ]:
#nuevo dataframe con las columnas seleccionadas
clean_data = data[columns]


In [ ]:
clean_data.shape

In [ ]:
#Guardando nuevo dataframe en un archivo csv
clean_data.to_csv('datasetLimpio.csv', index=False)